# Notebook 03 Data Wrangling & Preprocessing: Student 1M (Burnout & Fatigue)
**Proyek Analisis Data: HAPI (Human Activity Pattern Intelligence)**

### Profil Proyek & Tim
- **Tema Capstone:** Healthy Lives & Well-being
- **Target User:** Mahasiswa (fokus pada aspek akademik & lingkungan belajar)
- **Tujuan Proyek:** Pengembangan *platform* Webapp untuk *mood management & tracker* serta diagnosis mandiri tingkat kelelahan (*fatigue*).
- **Tim DS:** Greycia Febrina Michelle (CDCC700D6X2644) & Khazel Hayfa Yosmi (CDCC308D6X0629)

### Tujuan Notebook
Melakukan proses Data Wrangling secara *end-to-end* (Gathering → Assessing → Cleaning) dan Preprocessing pada dataset Student Mental Health & Burnout (1 juta records). Langkah ini bertujuan untuk menghasilkan data yang bersih dan terstruktur untuk digunakan sebagai tulang punggung (*backbone*) dataset bagi Dashboard dan Fatigue Score Fusion.

### Peran Dataset
Dataset yang digunakan dalam tahapan ini bersumber dari Kaggle: [Student Health Dataset](https://www.kaggle.com/datasets/ayeshasiddiqa123/student-health/data).

Dataset ini menyediakan data riil berskala besar mengenai kondisi mental mahasiswa yang mencakup kebiasaan belajar, pola tidur, tekanan akademik, dan skor burnout. 

Data yang telah diproses nantinya akan digunakan sebagai data pelatihan utama (*training data*) untuk model prediksi burnout serta menjadi sumber visualisasi data interaktif pada Dashboard Streamlit di aplikasi HAPI.

### Kolom Target
- `burnout_score`: skor kontinu yang berfungsi sebagai target untuk model regresi.
- `risk_level`: kategori tingkat risiko (Low / Medium / High) yang berfungsi sebagai target untuk model klasifikasi.

### Referensi Penelitian
- Sahili, Z. A., Patras, I., & Purver, M. (2024). Multimodal machine learning in mental health: a survey of data, algorithms, and challenges. arXiv preprint arXiv:2407.16804.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / 'data').exists():
        break
    ROOT = ROOT.parent

RAW_PATH   = ROOT / 'data' / 'raw' / 'model_ready' / 'burnout_fatigue_dirty.csv'
CLEAN_PATH = ROOT / 'data' / 'clean' / 'model_ready' / 'burnout_fatigue_clean.csv'
PREP_PATH  = ROOT / 'data' / 'preprocessed' / 'burnout_fatigue_preprocessed.csv'

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
PREP_PATH.parent.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    'study_hours_per_day', 'sleep_hours', 'exam_pressure',
    'physical_activity', 'social_support', 'screen_time',
    'stress_level', 'academic_performance',
]

DROP_COLS = [
    'anxiety_score', 'depression_score', 'mental_health_index',
    'dropout_risk', 'internet_usage', 'financial_stress', 'family_expectation',
]

TARGET_COLS = ['burnout_score', 'risk_level']

print(f'ROOT     : {ROOT}')
print(f'File ada : {RAW_PATH.exists()}')
print('Setup selesai.')

ROOT     : d:\Proyek_Analisis_Burnout
File ada : True
Setup selesai.


**Dokumentasi Setup Data: Study Habit**  
**Proyek HAPI (Human Activity Pattern Intelligence)**

Kami membuat script ini untuk menyiapkan lingkungan kerja dalam memproses dataset perilaku harian mahasiswa. Kami perlu memastikan struktur folder dan pemilihan fitur berjalan konsisten sebelum masuk ke tahap analisis perilaku.

**Operasi Utama**

**Path Mapping**  
Kami menggunakan `pathlib` untuk mendeteksi direktori root secara otomatis. Ini sangat membantu agar script tetap berjalan mulus meski kami berpindah-pindah lingkungan kerja antara laptop masing-masing.

**Directory Setup**  
Kami mengatur otomatisasi pembuatan folder jika belum tersedia. Kami membagi alur data menjadi tiga tahap untuk memastikan data mentah tidak tercampur dengan hasil proses:

- `RAW_PATH`: Lokasi file `study_habits_dirty.csv`.
- `CLEAN_PATH` & `PREP_PATH`: Folder untuk menampung hasil pembersihan dan penyiapan fitur perilaku.

**Schema Definition**  
Kami memilah dataset menjadi tiga kategori kolom agar data lebih fokus untuk model Fusion:

- `FEATURE_COLS`: Berisi variabel perilaku seperti `study_hours_per_day`, `sleep_hours`, `physical_activity`, dan `social_support`.
- `DROP_COLS`: Kolom yang kami eliminasi karena tidak relevan dengan kebutuhan input model prediksi fatigue.
- `TARGET_COLS`: Fokus utama kami adalah `stress_level` sebagai variabel numerik target.

**Alur Kerja Data**

```text
study_habits_dirty.csv
          │
          ▼
      RAW_PATH
          │
          ▼
  Data Cleaning
          │
          ▼
     CLEAN_PATH
          │
          ▼
 Feature Engineering
          │
          ▼
     PREP_PATH
          │
          ▼
  Fatigue Fusion Model
```

**Catatan Teknis**

Kami mengatur konfigurasi `pandas` agar menampilkan seluruh kolom secara penuh. Kami juga membatasi presisi angka floating point menjadi **4 digit** agar data yang kami kirim ke AI Engineer sudah dalam format yang rapi dan siap diolah ke tahap Fusion Model.

## Gathering Data

In [2]:
print('Memuat dataset...')
df_raw = pd.read_csv(RAW_PATH)
print(f'Dataset dimuat: {df_raw.shape[0]:,} baris, {df_raw.shape[1]} kolom')
print(f'Kolom: {df_raw.columns.tolist()}')
df_raw.head()

Memuat dataset...
Dataset dimuat: 1,015,900 baris, 21 kolom
Kolom: ['age', 'gender', 'academic_year', 'study_hours_per_day', 'exam_pressure', 'academic_performance', 'stress_level', 'anxiety_score', 'depression_score', 'sleep_hours', 'physical_activity', 'social_support', 'screen_time', 'internet_usage', 'financial_stress', 'family_expectation', 'burnout_score', 'mental_health_index', 'risk_level', 'dropout_risk', 'row_status']


,age,gender,academic_year,study_hours_per_day,exam_pressure,academic_performance,stress_level,anxiety_score,depression_score,sleep_hours,physical_activity,social_support,screen_time,internet_usage,financial_stress,family_expectation,burnout_score,mental_health_index,risk_level,dropout_risk,row_status
0,26,Female,4,4.0392,4.8452,73.9331,4.7260,2.9032,1.0017,6.0866,2.9110,4.9299,6.5263,7.5672,5.3650,3.0266,0.0000,6.9381,Low,0.4553,clean
1,28,Female,3,5.4805,6.7596,65.6219,5.9604,3.0631,2.4714,7.3321,2.6266,4.1431,8.8652,9.6255,6.1405,4.6207,2.6284,5.9555,Low,1.6355,clean
2,20,Male,4,5.2890,4.7802,77.3179,4.2223,3.2911,0.1658,4.9336,2.9162,6.6537,6.0086,5.4599,4.9806,8.1856,1.6349,7.2740,Low,0.1475,clean
3,17,Male,1,6.0913,7.8507,70.0237,6.2257,2.1698,3.4375,7.2808,2.3792,2.5976,8.2519,10.2637,5.1731,8.9323,3.2400,5.8275,Medium,2.9395,clean
4,23,Male,3,3.5938,3.0103,76.9082,0.6554,0.0000,0.0000,7.2766,1.8621,7.6323,5.3743,6.0599,6.1133,3.7356,0.0000,9.7378,Low,0.0000,clean


**Pemuatan Dataset Kesehatan Mental**

**Penjelasan Singkat**

- `pd.read_csv(RAW_PATH)`: Membaca file dataset utama ke dalam memori.

- `df_raw.shape`: Memeriksa dimensi dataset untuk mengetahui total volume data.

- `df_raw.columns.tolist()`: Mengambil daftar nama kolom guna memastikan semua variabel riset tersedia.

- `df_raw.head()`: Menampilkan lima baris pertama agar kami bisa memverifikasi format data secara visual.

**Ringkasan Data**

Hasil pemuatan menunjukkan dataset skala besar yang terdiri dari **1.015.900 baris** dan **21 kolom**. Variabel yang tersedia sangat komprehensif, mencakup demografi, metrik akademik, hingga indikator kesehatan mental seperti skor stres, kecemasan, dan depresi. Dataset ini tampak bersih dan siap digunakan untuk proses analisis lanjutan atau pemodelan prediksi risiko putus kuliah.

## Assessing Data

In [3]:
print('Shape')
print(f'Baris: {df_raw.shape[0]:,} | Kolom: {df_raw.shape[1]}')

Shape
Baris: 1,015,900 | Kolom: 21


In [4]:
print('Tipe Data')
print(df_raw.dtypes)

Tipe Data
age                       int64
gender                   object
academic_year             int64
study_hours_per_day     float64
exam_pressure           float64
academic_performance    float64
stress_level            float64
anxiety_score           float64
depression_score        float64
sleep_hours             float64
physical_activity       float64
social_support          float64
screen_time             float64
internet_usage          float64
financial_stress        float64
family_expectation      float64
burnout_score           float64
mental_health_index     float64
risk_level               object
dropout_risk            float64
row_status               object
dtype: object


In [5]:
print('Missing Values')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
mv_df = pd.DataFrame({'jumlah': missing, 'persen': missing_pct})
print(mv_df[mv_df['jumlah'] > 0] if mv_df['jumlah'].sum() > 0 else 'Tidak ada missing values.')

Missing Values
                     jumlah  persen
study_hours_per_day    3000  0.3000
sleep_hours            2500  0.2500
burnout_score          2000  0.2000
risk_level             1500  0.1500


In [6]:
print('Duplikat')
n_dup = df_raw.duplicated().sum()
print(f'Baris duplikat: {n_dup:,}')

Duplikat
Baris duplikat: 0


In [7]:
print('Statistik Deskriptif')
print(df_raw.describe().round(3))

Statistik Deskriptif
               age  academic_year  study_hours_per_day  exam_pressure  \
count 1015900.0000   1015900.0000         1012900.0000   1015900.0000   
mean       22.9960         2.5010               5.0160         5.9990   
std         3.7380         1.1180               2.0450         1.5480   
min        17.0000         1.0000               0.0000         1.0000   
25%        20.0000         2.0000               3.6510         4.9450   
50%        23.0000         3.0000               5.0000         5.9990   
75%        26.0000         3.0000               6.3500         7.0520   
max        29.0000         4.0000              23.9900        10.0000   

       academic_performance  stress_level  anxiety_score  depression_score  \
count          1015900.0000  1015900.0000   1015900.0000      1015900.0000   
mean                70.9990        4.2470         2.9870            1.2760   
std                  5.6610        1.6790         1.5090            1.2190   
min      

In [8]:
print('Distribusi Target')
print(f'Burnout Score — min: {df_raw["burnout_score"].min():.3f}, '
      f'max: {df_raw["burnout_score"].max():.3f}, '
      f'mean: {df_raw["burnout_score"].mean():.3f}')

print('\nRisk Level')
print(df_raw['risk_level'].value_counts(dropna=False))

print('\nDistribusi Row Status (Audit Masalah)')
print(df_raw['row_status'].value_counts())

Distribusi Target
Burnout Score — min: 0.000, max: 19.996, mean: 1.794

Risk Level
risk_level
Low         776162
Medium      220970
High         15268
NaN           1500
LOW            526
Lo             513
low            492
medium         128
Moderate       109
MEDIUM         103
Med             97
high            12
HIGH            11
Hi               9
Name: count, dtype: int64

Distribusi Row Status (Audit Masalah)
row_status
clean                                 1000000
dirty_missing_study_hours                3000
dirty_missing_sleep_hours                2500
dirty_typo_gender                        2000
dirty_typo_risk_level                    2000
dirty_missing_burnout_score              2000
dirty_missing_risk_level                 1500
dirty_outlier_study_hours_high           1000
dirty_outlier_sleep_hours_negative        800
dirty_outlier_burnout_score_high          600
dirty_duplicate                           500
Name: count, dtype: int64


In [9]:
print('Cek Potensi Leakage')
print('Korelasi kolom numerik terhadap burnout_score (dari baris clean):')

df_clean_check = df_raw[df_raw['row_status'] == 'clean']
num_cols = df_clean_check.select_dtypes(include=[np.number]).columns

corr = (
    df_clean_check[num_cols]
    .corr()['burnout_score']
    .drop('burnout_score')
    .abs()
    .sort_values(ascending=False)
)

print(corr.round(4))

print('\nKolom dengan korelasi > 0.85 dengan burnout_score (kandidat leakage):')
print(corr[corr > 0.85].index.tolist())

Cek Potensi Leakage
Korelasi kolom numerik terhadap burnout_score (dari baris clean):
mental_health_index    0.7965
stress_level           0.7531
dropout_risk           0.6897
anxiety_score          0.6685
depression_score       0.6426
exam_pressure          0.4344
sleep_hours            0.3714
study_hours_per_day    0.3351
financial_stress       0.2957
social_support         0.2298
family_expectation     0.2178
physical_activity      0.1101
academic_performance   0.0571
screen_time            0.0012
internet_usage         0.0012
age                    0.0010
academic_year          0.0006
Name: burnout_score, dtype: float64

Kolom dengan korelasi > 0.85 dengan burnout_score (kandidat leakage):
[]


In [10]:
print('Outlier Check (IQR) pada fitur utama')

df_clean_check = df_raw[df_raw['row_status'] == 'clean']

for col in ['study_hours_per_day', 'sleep_hours', 'burnout_score']:
    if col in df_clean_check.columns:
        Q1 = df_clean_check[col].quantile(0.25)
        Q3 = df_clean_check[col].quantile(0.75)
        IQR = Q3 - Q1
        n_out = (
            (df_clean_check[col] < Q1 - 1.5 * IQR) |
            (df_clean_check[col] > Q3 + 1.5 * IQR)
        ).sum()

        print(f'{col}: {n_out:,} outlier ({n_out/len(df_clean_check)*100:.2f}%)')

Outlier Check (IQR) pada fitur utama
study_hours_per_day: 3,491 outlier (0.35%)
sleep_hours: 0 outlier (0.00%)
burnout_score: 3,735 outlier (0.37%)


**Profil Dataset**

Kami mengolah **1.015.900 entri** dengan **21 kolom** yang mencakup profil demografi, pola aktivitas harian, hingga indikator kesehatan mental. Secara struktural, sebagian besar tipe data sudah terbaca dengan benar, namun skala data yang masif menuntut perhatian ekstra pada pembersihan data agar kualitas model tetap terjaga.

**Temuan Kritis**

Kami mengidentifikasi beberapa titik masalah yang akan mengganggu hasil analisis jika dibiarkan:

- **Data Kosong:** Terdapat celah pada variabel penting seperti `study_hours_per_day`, `sleep_hours`, `burnout_score`, dan `risk_level` dengan total ribuan baris yang terdampak.

- **Inkonsistensi Kategori:** Label `risk_level` memiliki variasi penulisan yang berantakan, seperti `Lo`, `low`, `MEDIUM`, dan `Moderate`, yang harus segera diseragamkan ke kategori baku.

- **Anomali Logika:** Kami mendeteksi adanya durasi tidur negatif dan nilai jam belajar yang tidak masuk akal (di atas 23 jam per hari), yang merupakan tanda kesalahan input data atau sistem pencatatan.

- **Potensi Leakage:** Setelah melakukan uji korelasi, tidak ditemukan variabel dengan korelasi di atas 0.85 terhadap `burnout_score`, sehingga risiko kebocoran data (data leakage) saat pemodelan terlihat rendah.

**Ringkasan Hasil Assessing**

| Masalah | Detail | Tindakan |
|----------|----------|----------|
| Missing values | study_hours, sleep_hours, burnout, risk_level | Hapus baris dengan nilai kosong (proporsi kecil terhadap total data) |
| Outlier/invalid | Jam tidur negatif, jam belajar > 24, outlier statistik | Hapus baris agar data tetap valid secara logis |
| Typo risk_level | 'Lo', 'low', 'MEDIUM', 'Moderate', 'Hi' | Koreksi ke kategori standar (Low, Medium, High) |
| Duplikat | 500 baris duplikat terdeteksi | Hapus baris |
| Kolom tidak diperlukan | row_status | Drop sebelum masuk ke tahap pelatihan model |

## Cleaning Data

In [11]:
df = df_raw.copy()
before = len(df)

df = df[df['row_status'] == 'clean'].copy()
print(f'Drop Baris Kotor: {before - len(df):,} dihapus → sisa {len(df):,}')

Drop Baris Kotor: 15,900 dihapus → sisa 1,000,000


In [12]:
df = df.drop(columns=['row_status'])
print('Drop Kolom Row Status ✓')
print(f'Missing Values Tersisa: {df.isnull().sum().sum()}')

Drop Kolom Row Status ✓
Missing Values Tersisa: 0


In [13]:
cols_to_drop = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=cols_to_drop)

print(f'Drop Kolom: {cols_to_drop}')
print(f'Kolom Tersisa: {df.columns.tolist()}')

Drop Kolom: ['anxiety_score', 'depression_score', 'mental_health_index', 'dropout_risk', 'internet_usage', 'financial_stress', 'family_expectation']
Kolom Tersisa: ['age', 'gender', 'academic_year', 'study_hours_per_day', 'exam_pressure', 'academic_performance', 'stress_level', 'sleep_hours', 'physical_activity', 'social_support', 'screen_time', 'burnout_score', 'risk_level']


In [14]:
if 'study_hours_per_day' in df.columns:
    n = ((df['study_hours_per_day'] < 0) | (df['study_hours_per_day'] > 14)).sum()
    df['study_hours_per_day'] = df['study_hours_per_day'].clip(0, 14)
    print(f'Clip study_hours_per_day ke [0,14]: {n} nilai')

if 'sleep_hours' in df.columns:
    n = ((df['sleep_hours'] < 0) | (df['sleep_hours'] > 14)).sum()
    df['sleep_hours'] = df['sleep_hours'].clip(0, 14)
    print(f'Clip sleep_hours ke [0,14]: {n} nilai')

if 'burnout_score' in df.columns:
    n = ((df['burnout_score'] < 0) | (df['burnout_score'] > 10)).sum()
    df['burnout_score'] = df['burnout_score'].clip(0, 10)
    print(f'Clip burnout_score ke [0,10]: {n} nilai')

valid_gender = {'Male', 'Female', 'Other'}
valid_risk   = {'Low', 'Medium', 'High'}

n_bad_gender = df[~df['gender'].isin(valid_gender)].shape[0]
n_bad_risk   = df[~df['risk_level'].isin(valid_risk)].shape[0]

assert n_bad_gender == 0, f'gender tidak valid: {df["gender"].unique()}'
assert n_bad_risk == 0, f'risk_level tidak valid: {df["risk_level"].unique()}'

print('Gender Valid ✓')
print('Risk Level Valid ✓')

assert df.isnull().sum().sum() == 0, 'Masih ada missing values!'
print('Missing Values: 0 ✓')

Clip study_hours_per_day ke [0,14]: 0 nilai
Clip sleep_hours ke [0,14]: 0 nilai
Clip burnout_score ke [0,10]: 0 nilai
Gender Valid ✓
Risk Level Valid ✓
Missing Values: 0 ✓


In [15]:
print(f'Shape Akhir Cleaned: {df.shape}')
print(f'Missing Values: {df.isnull().sum().sum()}')
df.head(3)

Shape Akhir Cleaned: (1000000, 13)
Missing Values: 0


,age,gender,academic_year,study_hours_per_day,exam_pressure,academic_performance,stress_level,sleep_hours,physical_activity,social_support,screen_time,burnout_score,risk_level
0,26,Female,4,4.0392,4.8452,73.9331,4.7260,6.0866,2.9110,4.9299,6.5263,0.0000,Low
1,28,Female,3,5.4805,6.7596,65.6219,5.9604,7.3321,2.6266,4.1431,8.8652,2.6284,Low
2,20,Male,4,5.2890,4.7802,77.3179,4.2223,4.9336,2.9162,6.6537,6.0086,1.6349,Low


In [16]:
df.to_csv(CLEAN_PATH, index=False)
print(f'Dataset cleaned disimpan ke: {CLEAN_PATH}')

Dataset cleaned disimpan ke: d:\Proyek_Analisis_Burnout\data\clean\model_ready\burnout_fatigue_clean.csv


**Pembersihan Data untuk Dataset Burnout Utama**

Kami merampungkan tahap pembersihan data skala besar untuk memastikan kualitas input sebelum masuk ke pemodelan prediktif. Berikut rangkaian proses yang kami kerjakan.

**Langkah-Langkah Teknis**

- **Penyaringan Data Mentah:** Kami membuang **15.900 baris** yang tidak valid agar data yang diproses hanya data dengan status clean.

- **Pemangkasan Kolom:** Kami menghapus kolom `row_status` dan serangkaian variabel lain yang tidak relevan dengan target fitur untuk menyederhanakan dataset.

- **Normalisasi Rentang Nilai:** Kami menerapkan clipping pada kolom `study_hours_per_day` dan `sleep_hours` ke rentang **0 hingga 14**, serta `burnout_score` ke rentang **0 hingga 10** untuk menjaga konsistensi logis data.

- **Validasi Integritas:** Kami melakukan pengecekan ketat untuk memastikan variabel kategorikal seperti `gender` dan `risk_level` hanya berisi label yang valid.

- **Verifikasi Akhir:** Kami memastikan tidak ada missing values yang tersisa pada seluruh kolom setelah proses pembersihan selesai.

**Hasil Akhir**

Setelah seluruh proses tadi, kami menghasilkan dataset bersih dengan total **1.000.000 baris** dan **13 kolom** siap pakai. Data final ini kami simpan ke dalam file `burnout_fatigue_clean.csv` di direktori yang sudah ditentukan.

## Preprocessing & Feature Engineering

In [17]:
df_prep = pd.read_csv(CLEAN_PATH)

if 'study_hours_per_day' in df_prep.columns and 'sleep_hours' in df_prep.columns:
    total = df_prep['study_hours_per_day'] + df_prep['sleep_hours']
    df_prep['study_sleep_ratio'] = (
        df_prep['study_hours_per_day'] / total.replace(0, np.nan)
    ).fillna(0).clip(0, 1).round(4)

    print(f'study_sleep_ratio — min: {df_prep["study_sleep_ratio"].min():.4f}, '
          f'max: {df_prep["study_sleep_ratio"].max():.4f}')

study_sleep_ratio — min: 0.0000, max: 0.8064


In [18]:
pressure_cols = [c for c in ['exam_pressure', 'stress_level'] if c in df_prep.columns]

if pressure_cols:
    df_prep['academic_pressure_index'] = df_prep[pressure_cols].mean(axis=1).round(4)
    print(f'academic_pressure_index dari {pressure_cols}')

academic_pressure_index dari ['exam_pressure', 'stress_level']


### Encoding

In [19]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df_prep.select_dtypes(include='object').columns.tolist()
cat_cols_excl = [c for c in cat_cols if c not in ['risk_level']]

le = LabelEncoder()

for col in cat_cols_excl:
    df_prep[f'{col}_encoded'] = le.fit_transform(df_prep[col].astype(str))
    print(f'ENC {col} → {col}_encoded: {dict(zip(le.classes_, le.transform(le.classes_)))}')

ENC gender → gender_encoded: {'Female': np.int64(0), 'Male': np.int64(1), 'Other': np.int64(2)}


In [20]:
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_prep['risk_level_encoded'] = df_prep['risk_level'].map(risk_map)

n_null = df_prep['risk_level_encoded'].isnull().sum()
assert n_null == 0, f'{n_null} nilai gagal di-encode — cek nilai unik risk_level'

print(f'ENC risk_level: {risk_map}')
print(df_prep[['risk_level', 'risk_level_encoded']].value_counts().sort_index())

ENC risk_level: {'Low': 0, 'Medium': 1, 'High': 2}
risk_level  risk_level_encoded
High        2                      15080
Low         0                     766645
Medium      1                     218275
Name: count, dtype: int64


### Leakage Check

In [21]:
all_features = [c for c in df_prep.columns if c not in ['burnout_score', 'risk_level', 'risk_level_encoded']]
target_check = set(['burnout_score', 'risk_level', 'risk_level_encoded'])
leakage = set(all_features) & target_check
assert len(leakage) == 0, f'LEAKAGE: {leakage}'
print(f'Leakage check: BERSIH ✓')

Leakage check: BERSIH ✓


### Finalisasi

In [22]:
df_prep.to_csv(PREP_PATH, index=False)
print(f'Dataset model-ready disimpan ke: {PREP_PATH}')
print(f'\n=== RINGKASAN FINAL ===')
print(f'Total baris         : {len(df_prep):,}')
print(f'Total kolom         : {df_prep.shape[1]}')
print(f'Missing values      : {df_prep.isnull().sum().sum()}')
print(f'Leakage             : BERSIH')
print(f'\nKolom target:')
print(f'  burnout_score       → regresi (0.0–10.0)')
print(f'  risk_level          → klasifikasi (Low/Medium/High)')
print(f'  risk_level_encoded  → klasifikasi numerik (0/1/2)')
print(f'\nDistribusi risk_level:')
print(df_prep['risk_level'].value_counts())

Dataset model-ready disimpan ke: d:\Proyek_Analisis_Burnout\data\preprocessed\burnout_fatigue_preprocessed.csv

=== RINGKASAN FINAL ===
Total baris         : 1,000,000
Total kolom         : 17
Missing values      : 0
Leakage             : BERSIH

Kolom target:
  burnout_score       → regresi (0.0–10.0)
  risk_level          → klasifikasi (Low/Medium/High)
  risk_level_encoded  → klasifikasi numerik (0/1/2)

Distribusi risk_level:
risk_level
Low       766645
Medium    218275
High       15080
Name: count, dtype: int64


**Rekayasa Fitur Lanjutan untuk Pemodelan Burnout**

Kami telah menuntaskan tahap rekayasa fitur agar dataset siap diolah oleh model machine learning. Fokus kami adalah memperkaya data melalui variabel turunan serta memastikan seluruh data kategorikal terkonversi ke format numerik yang dipahami mesin.

**Verifikasi dan Transformasi Data**

- **Fitur Turunan:** Kami membuat `study_sleep_ratio` untuk melihat keseimbangan waktu belajar dan istirahat, serta `academic_pressure_index` yang menggabungkan skor tekanan ujian dan tingkat stres sebagai satu indikator komposit.

- **Encoding Data:** Seluruh variabel kategorikal, seperti `gender`, kami ubah menggunakan `LabelEncoder`. Sementara itu, label `risk_level` dikonversi secara manual ke format numerik (`0`, `1`, `2`) untuk keperluan pemodelan klasifikasi yang presisi.

- **Validasi Encoding:** Kami melakukan pengecekan ulang terhadap hasil konversi label guna memastikan tidak ada data yang gagal terproses dan semua kategori sudah terpetakan dengan benar.

**Jaminan Kualitas**

Kami menerapkan beberapa prosedur untuk mencegah kegagalan model di masa depan:

- **Cek Kebocoran Data:** Kami memastikan variabel target (`burnout_score` dan `risk_level`) benar benar terpisah dari fitur input. Langkah ini krusial agar model tidak melakukan prediksi berdasarkan informasi yang seharusnya belum tersedia.

- **Struktur Final:** Dataset kini memiliki **17 kolom** yang mencakup data dasar, fitur hasil rekayasa, serta target numerik maupun kategorikal.

- **Penyimpanan:** File hasil akhir sudah tersimpan di `burnout_fatigue_preprocessed.csv`.

**Ringkasan**

Ringkasnya, kami sudah memiliki **1.000.000 baris** data yang bersih dan kaya fitur. Dataset ini siap untuk digunakan dalam berbagai eksperimen model regresi maupun klasifikasi. Langkah berikutnya adalah masuk ke fase training untuk melihat performa model dalam mengenali risiko burnout.